# Stage 4 L0 — Layer mapping for Qwen3.6-35B-A3B on SuperGPQA

**Goal**: before spending compute on SAE training, find the layer where the model's residual stream most strongly separates correct vs wrong responses on SuperGPQA. This is the peak contrastive layer — our SAE target.

**Method**:
1. Load Qwen3.6-35B-A3B (bf16, 96 GB GPU — fits with margin)
2. Generate greedy responses with thinking mode on 100 held-out SuperGPQA questions
3. Verify correct/wrong via letter match against `answer_letter`
4. Hook all 40 layers' residual stream, capture mean over response tokens
5. Compute top-10 Cohen's d per layer across 2048 hidden dims
6. Pick peak layer; compare GDN (mostly) vs Gated Attn (layers 3, 7, ..., 39)

**Budget**: ~2 h on RTX 6000 Blackwell 96 GB

**Output**: peak layer index + plot of contrastive effect size per layer.

## Cell 1 — Install (same stack as G3: pinned transformers, hf_hub==1.5.0, fla, NO numpy pin)

In [ ]:
import sys, subprocess, os, shutil, site
from pathlib import Path

def _pip(*args, check=True):
    return subprocess.run([sys.executable, '-m', 'pip', *args], check=check)

def _check_qwen36():
    try:
        import transformers
        try:
            from transformers.models.auto.auto_mappings import CONFIG_MAPPING_NAMES
        except ImportError:
            from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
        # NOTE: Qwen3.6-35B-A3B internally uses model_type 'qwen3_5_moe' (same architecture family).
        # The '3.6' is branding only.
        has_it = 'qwen3_5_moe' in CONFIG_MAPPING_NAMES
        return has_it, transformers.__version__
    except Exception as e:
        return False, f'import-error: {e}'

ok, ver = _check_qwen36()
print(f'Current: transformers={ver}, qwen3_x supported={ok}')

# If qwen3_6 isn't registered yet, we'll need a newer commit than the Qwen3.5 pin.
# Try main HEAD; if it breaks, fall back to 0aff0dbf0aa1 (known Qwen3.5) and pray qwen3_6 is registered.
TRANSFORMERS_REF = 'main'  # change to a commit SHA if main is broken
SRC_DIR = '/content/transformers_src'

if not ok:
    print(f'\n→ Installing transformers @ {TRANSFORMERS_REF}')
    _pip('install', '-q',
         'accelerate', 'peft', 'datasets',
         'huggingface_hub==1.5.0',
         'safetensors', 'sympy', 'einops', 'scikit-learn', 'matplotlib',
         'sentencepiece', 'tokenizers', 'protobuf')
    _pip('uninstall', '-y', 'transformers', check=False)
    _pip('uninstall', '-y', 'transformers', check=False)
    for p in sys.path + site.getsitepackages() + [site.getusersitepackages()]:
        tr = Path(p) / 'transformers'
        if tr.exists():
            shutil.rmtree(tr, ignore_errors=True)
    if os.path.exists(SRC_DIR):
        shutil.rmtree(SRC_DIR)
    subprocess.run(['git', 'clone', '--quiet', 'https://github.com/huggingface/transformers.git', SRC_DIR], check=True)
    if TRANSFORMERS_REF != 'main':
        subprocess.run(['git', '-C', SRC_DIR, 'checkout', '--quiet', TRANSFORMERS_REF], check=True)
    _pip('install', '--force-reinstall', '--no-deps', '--no-cache-dir', SRC_DIR)
    _pip('install', '--no-cache-dir', 'flash-linear-attention', 'causal-conv1d', check=False)

    for mod in list(sys.modules):
        if mod.startswith('transformers') or mod.startswith('huggingface_hub'):
            del sys.modules[mod]
    ok, ver = _check_qwen36()
    print(f'Post-install: transformers={ver}, qwen3_x supported={ok}')
    if not ok:
        print('\n⚠️  Restart required. Go to Runtime > Restart session, then rerun this cell.')
        raise SystemExit

# Verify qwen3_6 is registered (may need newer transformers than our Qwen3.5 pin)
try:
    from transformers.models.auto.auto_mappings import CONFIG_MAPPING_NAMES
except ImportError:
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
print(f'qwen3_5_moe registered: {"qwen3_5_moe" in CONFIG_MAPPING_NAMES}')
print('(Qwen3.6-35B-A3B internally is qwen3_5_moe — branding only.)')

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        print('✅ HF authenticated')
except Exception as e:
    print(f'(HF auth skipped: {e})')

from google.colab import drive
drive.mount('/content/drive')

import json, math, time, random, re, gc
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

DRIVE_ROOT = Path('/content/drive/MyDrive/mechreward/s4_qwen36')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(f'\nDrive root: {DRIVE_ROOT}')

## Cell 2 — CFG

In [ ]:
CFG = dict(
    model_id='Qwen/Qwen3.6-35B-A3B',
    n_questions=100,                      # SuperGPQA held-out probe set
    difficulty_filter=['middle', 'hard'], # skip 'easy' to stay informative (~65 % baseline on overall)
    max_new_tokens=2048,                  # thinking mode needs headroom
    n_layers=40,                          # Qwen3.6-35B-A3B
    d_model=2048,
    gated_attn_layers=[3, 7, 11, 15, 19, 23, 27, 31, 35, 39],
    top_k_dims=10,                        # Cohen's d aggregation
    seed=42,
    cache_dir=str(DRIVE_ROOT),
)
print(json.dumps(CFG, indent=2))

## Cell 3 — Load model (bf16, hopefully fits 96 GB; fallback to INT8 via bnb)

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText

tok = AutoTokenizer.from_pretrained(CFG['model_id'], trust_remote_code=True)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token

# Try bf16 first (70 GB weights; fits on 96 GB with room for activations)
model = AutoModelForImageTextToText.from_pretrained(
    CFG['model_id'],
    dtype=torch.bfloat16,
    device_map='cuda',
    attn_implementation='sdpa',
    trust_remote_code=True,
)
model.eval()
print(f'Model loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')

## Cell 4 — Locate layers (same helper as Qwen3.5; verify path)

In [ ]:
def get_layer_module(m, idx):
    base = m.base_model.model if hasattr(m, 'base_model') else m
    for path in [('model','language_model','layers'),
                 ('language_model','layers'),
                 ('model','layers'),
                 ('model','model','layers'),
                 ('layers',)]:
        cur = base; ok = True
        for p in path:
            if hasattr(cur, p): cur = getattr(cur, p)
            else: ok = False; break
        if ok and hasattr(cur, '__getitem__'):
            try: return cur[idx]
            except Exception: continue
    raise RuntimeError('layers not located')

# Sanity
for i in [0, 3, 7, 19, 27, 39]:
    layer = get_layer_module(model, i)
    layer_type = type(layer).__name__
    is_ga = i in CFG['gated_attn_layers']
    print(f'L{i:2d} [{("GatedAttn" if is_ga else "GDN"):9s}]: {layer_type}')

## Cell 5 — Dataset (SuperGPQA probe set)

In [ ]:
from datasets import load_dataset

raw = load_dataset('m-a-p/SuperGPQA', split='train')  # only has train split
print(f'Total: {len(raw)}')

# Filter by difficulty
filtered = raw.filter(lambda ex: ex['difficulty'] in CFG['difficulty_filter'])
print(f'Filtered ({"+".join(CFG["difficulty_filter"])}): {len(filtered)}')

# Sample deterministic probe set
probe = filtered.shuffle(seed=CFG['seed']).select(range(CFG['n_questions']))
print(f'Probe set: {len(probe)}')
print(f'\nExample:')
ex = probe[0]
print(f'  discipline: {ex["discipline"]}/{ex["field"]}')
print(f'  difficulty: {ex["difficulty"]}  is_calculation: {ex["is_calculation"]}')
print(f'  Q: {ex["question"][:200]}')
print(f'  #options: {len(ex["options"])}, gold letter: {ex["answer_letter"]}')

## Cell 6 — Prompt + verify helpers

In [ ]:
def format_prompt(ex):
    """Multiple-choice prompt. Answer format: letter after 'Answer:' or in \\boxed{}."""
    choices = '\n'.join(f'{chr(65+i)}. {opt}' for i, opt in enumerate(ex['options']))
    msgs = [{
        'role': 'user',
        'content': (
            f"Answer the following multiple-choice question. Reason step by step, "
            f"then put the letter of the correct answer in \\boxed{{}}.\n\n"
            f"Question: {ex['question']}\n\n"
            f"Options:\n{choices}"
        )
    }]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

BOXED_RE = re.compile(r'\\boxed\{\s*([A-J])\s*\}')
LETTER_AT_END_RE = re.compile(r'[Aa]nswer[:\s]+([A-J])\b')

def extract_letter(completion, n_opts):
    m = list(BOXED_RE.finditer(completion))
    if m:
        letter = m[-1].group(1).upper()
        if ord(letter) - ord('A') < n_opts:
            return letter
    m2 = list(LETTER_AT_END_RE.finditer(completion))
    if m2:
        letter = m2[-1].group(1).upper()
        if ord(letter) - ord('A') < n_opts:
            return letter
    # Fallback: last standalone letter
    tail = completion[-200:]
    cands = re.findall(r'\b([A-J])\b', tail)
    if cands:
        letter = cands[-1]
        if ord(letter) - ord('A') < n_opts:
            return letter
    return None

def verify(ex, completion):
    pred = extract_letter(completion, len(ex['options']))
    return pred is not None and pred == ex['answer_letter']

# Test extraction on a small sample
enc = tok(format_prompt(probe[0]), return_tensors='pt').to('cuda')
with torch.no_grad():
    test_out = model.generate(**enc, max_new_tokens=512, do_sample=False, pad_token_id=tok.pad_token_id)
test_completion = tok.decode(test_out[0][enc.input_ids.shape[1]:], skip_special_tokens=True)
print(f'Gold: {probe[0]["answer_letter"]}')
print(f'Pred: {extract_letter(test_completion, len(probe[0]["options"]))}')
print(f'Completion tail: ...{test_completion[-400:]}')

## Cell 7 — Generate + capture all-layer residuals

Registers 40 forward hooks, one per layer. Captures the mean of the response-token hidden states at each layer. Then generates greedy with thinking mode.

In [ ]:
class MultiLayerCapture:
    def __init__(self, n_layers):
        self.n_layers = n_layers
        self.hiddens = {i: None for i in range(n_layers)}

    def make_hook(self, idx):
        def hook(module, inp, out):
            h = out[0] if isinstance(out, tuple) else out
            self.hiddens[idx] = h.detach()
        return hook

    def reset(self):
        for i in range(self.n_layers):
            self.hiddens[i] = None

def register_all_hooks(m, n_layers):
    cap = MultiLayerCapture(n_layers)
    handles = []
    for i in range(n_layers):
        h = get_layer_module(m, i).register_forward_hook(cap.make_hook(i))
        handles.append(h)
    return cap, handles

from tqdm.auto import tqdm

per_layer_resp_means = {i: [] for i in range(CFG['n_layers'])}  # list of [d_model] tensors per Q
outcomes = []
gold_letters = []
pred_letters = []
n_tok_gen = []

model.config.use_cache = True  # generate fast
cap, handles = register_all_hooks(model, CFG['n_layers'])

t0 = time.time()
try:
    for qi, ex in enumerate(tqdm(probe, desc='Generating + capturing')):
        prompt = format_prompt(ex)
        enc = tok(prompt, return_tensors='pt').to('cuda')
        P = enc.input_ids.shape[1]
        with torch.no_grad():
            gen = model.generate(
                **enc,
                max_new_tokens=CFG['max_new_tokens'],
                do_sample=False,
                pad_token_id=tok.pad_token_id,
                use_cache=True,
            )
        # Second forward on full sequence to capture ALL-position hidden states at all layers
        cap.reset()
        attn_mask = (gen != tok.pad_token_id).long()
        with torch.no_grad():
            model(input_ids=gen, attention_mask=attn_mask, use_cache=False)
        # For each layer, take mean over response tokens (position >= P, attn=1)
        response_mask = attn_mask.clone()
        response_mask[:, :P] = 0
        n_resp = response_mask.sum().item()
        for i in range(CFG['n_layers']):
            h = cap.hiddens[i][0]  # [T, d]
            masked = h * response_mask[0].unsqueeze(-1).to(h.dtype)
            pooled = masked.sum(dim=0) / max(n_resp, 1)
            per_layer_resp_means[i].append(pooled.float().cpu())

        # Verify
        completion = tok.decode(gen[0][P:], skip_special_tokens=True)
        is_correct = verify(ex, completion)
        outcomes.append(int(is_correct))
        gold_letters.append(ex['answer_letter'])
        pred_letters.append(extract_letter(completion, len(ex['options'])))
        n_tok_gen.append(gen.shape[1] - P)
finally:
    for h in handles: h.remove()

outcomes = np.array(outcomes)
dt_total = time.time() - t0
print(f'\nTotal time: {dt_total/60:.1f} min')
print(f'Correct: {outcomes.sum()} / {len(outcomes)} = {outcomes.mean()*100:.1f}%')
print(f'Avg tokens generated: {np.mean(n_tok_gen):.0f}  (max {np.max(n_tok_gen)})')
print(f'Capture shape per layer: {len(per_layer_resp_means[0])} questions × {per_layer_resp_means[0][0].shape[0]} dim')

# Save raw captures for later re-analysis without re-running generation
save_obj = {
    'per_layer_means': {i: torch.stack(per_layer_resp_means[i]) for i in range(CFG['n_layers'])},
    'outcomes': outcomes,
    'gold_letters': gold_letters,
    'pred_letters': pred_letters,
    'n_tok_gen': n_tok_gen,
    'cfg': CFG,
}
torch.save(save_obj, DRIVE_ROOT / 'layer_mapping_captures.pt')
print(f'\u2705 Saved: {DRIVE_ROOT}/layer_mapping_captures.pt')

## Cell 8 — Compute Cohen's d per layer

In [ ]:
correct_mask = outcomes.astype(bool)
wrong_mask = ~correct_mask
print(f'n_correct={correct_mask.sum()}, n_wrong={wrong_mask.sum()}')

if correct_mask.sum() < 10 or wrong_mask.sum() < 10:
    print('\u26a0️  Very unbalanced correct/wrong split — Cohen\'s d will be noisy. Consider filtering dataset differently.')

layer_top10_d = []
layer_top10_mean_d = []
layer_full_d_vectors = {}

for i in range(CFG['n_layers']):
    X = save_obj['per_layer_means'][i].numpy()  # [n_q, d_model]
    h_c = X[correct_mask]
    h_w = X[wrong_mask]
    mean_c = h_c.mean(axis=0)
    mean_w = h_w.mean(axis=0)
    var_c = h_c.var(axis=0, ddof=1)
    var_w = h_w.var(axis=0, ddof=1)
    pooled_std = np.sqrt((var_c + var_w) / 2)
    d_per_dim = (mean_c - mean_w) / (pooled_std + 1e-8)
    abs_d = np.abs(d_per_dim)
    top10 = np.sort(abs_d)[::-1][:CFG['top_k_dims']]
    layer_top10_d.append(top10.sum())
    layer_top10_mean_d.append(top10.mean())
    layer_full_d_vectors[i] = d_per_dim

layer_top10_d = np.array(layer_top10_d)
layer_top10_mean_d = np.array(layer_top10_mean_d)

print('\nTop-10 |Cohen\'s d| sum per layer:')
for i in range(CFG['n_layers']):
    is_ga = i in CFG['gated_attn_layers']
    marker = ' ★ GA' if is_ga else ''
    print(f'  L{i:2d}: {layer_top10_d[i]:6.3f}  (mean |d| top-10 = {layer_top10_mean_d[i]:.3f}){marker}')

peak_layer = int(np.argmax(layer_top10_d))
print(f'\n🎯 Peak layer: L{peak_layer}  (top-10 |d| sum = {layer_top10_d[peak_layer]:.3f})')
print(f'   {"GATED ATTENTION" if peak_layer in CFG["gated_attn_layers"] else "GDN"}  |  depth {peak_layer/CFG["n_layers"]*100:.0f}%')

## Cell 9 — Visualize + pick SAE target

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
layers = np.arange(CFG['n_layers'])
ga_mask = np.array([i in CFG['gated_attn_layers'] for i in layers])

ax.bar(layers[~ga_mask], layer_top10_d[~ga_mask], color='#4A90E2', label='GDN layer', alpha=0.75)
ax.bar(layers[ga_mask], layer_top10_d[ga_mask], color='#E94B3C', label='Gated Attention layer')
ax.axvline(peak_layer, color='gold', lw=2, ls='--', alpha=0.8, label=f'peak (L{peak_layer})')
ax.set_xlabel('layer')
ax.set_ylabel('sum of top-10 |Cohen\'s d|')
ax.set_title(f'Qwen3.6-35B-A3B — contrastive effect size per layer (SuperGPQA, n={CFG["n_questions"]})')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(DRIVE_ROOT / 'layer_mapping.png', dpi=150, bbox_inches='tight')
plt.show()

# SAE target recommendation
print('\n=== SAE TARGET RECOMMENDATION ===')
print(f'Peak layer: L{peak_layer} ({"GatedAttn" if peak_layer in CFG["gated_attn_layers"] else "GDN"}, {peak_layer/CFG["n_layers"]*100:.0f}% depth)')

# If peak is GDN, train SAE right there OR 2-3 layers downstream (lesson from Qwen3.5: L13 peak, SAE at L18 worked well)
# If peak is Gated Attn, train there (attention layers consolidate info)
if peak_layer in CFG['gated_attn_layers']:
    sae_target = peak_layer
    print(f'→ Gated Attn peak: train SAE directly at L{sae_target}')
else:
    # Find next Gated Attn layer after peak as a structural anchor
    downstream_ga = [l for l in CFG['gated_attn_layers'] if l > peak_layer]
    if downstream_ga:
        sae_target = min(downstream_ga)
        print(f'→ GDN peak; nearest downstream Gated Attn is L{sae_target}. Train SAE there to combine peak signal with attn consolidation.')
    else:
        sae_target = peak_layer + 2
        print(f'→ GDN peak late; train SAE at L{sae_target} (peak + 2, like Qwen3.5 L13→L18 pattern).')

print(f'\n✅ RECOMMENDED SAE TARGET: L{sae_target}')

## Cell 10 — Logit-lens: when does English-math-reasoning commit?

In [ ]:
# Pick a few reasoning-relevant token IDs
reasoning_tokens = ['####', 'boxed', 'answer', 'Answer', 'Step', 'Therefore', 'So']
token_ids = {t: tok.encode(t, add_special_tokens=False) for t in reasoning_tokens}
# Use just the first token id for tokens that split into multiple
tid_flat = {t: ids[0] for t, ids in token_ids.items() if ids}
print(f'Reasoning tokens (first token id):')
for t, i in tid_flat.items():
    print(f'  {t!r}: id={i}')

# Get final lm_head weight
lm_head = None
for name in ['lm_head', 'model.lm_head']:
    parts = name.split('.')
    cur = model
    for p in parts:
        cur = getattr(cur, p, None)
        if cur is None: break
    if cur is not None:
        lm_head = cur
        break
assert lm_head is not None, 'could not find lm_head'
print(f'lm_head: {type(lm_head).__name__}, weight shape={tuple(lm_head.weight.shape)}')

# For each layer, project the mean hidden (across correct responses) through lm_head, get probability of reasoning tokens
layer_reasoning_prob = np.zeros(CFG['n_layers'])
W = lm_head.weight.to(torch.float32).cuda()  # [vocab, d_model]
for i in range(CFG['n_layers']):
    h_mean = save_obj['per_layer_means'][i].float().mean(dim=0).cuda()  # [d_model]
    logits = h_mean @ W.T  # [vocab]
    probs = torch.softmax(logits, dim=-1)
    p = sum(probs[tid_flat[t]].item() for t in tid_flat)
    layer_reasoning_prob[i] = p

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(np.arange(CFG['n_layers']), layer_reasoning_prob, marker='o', color='#E94B3C')
for ga in CFG['gated_attn_layers']:
    ax.axvline(ga, color='gray', alpha=0.2, ls=':')
ax.set_xlabel('layer')
ax.set_ylabel('sum P(reasoning tokens)')
ax.set_title(f'Logit lens — when does the model commit to English reasoning tokens? ({", ".join(tid_flat.keys())})')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(DRIVE_ROOT / 'logit_lens.png', dpi=150, bbox_inches='tight')
plt.show()

commit_layer = int(np.argmax(np.diff(layer_reasoning_prob)) + 1)
print(f'\nStrongest rise in reasoning-token prob between L{commit_layer-1} and L{commit_layer}')

## Cell 11 — Summary + save decision

In [ ]:
summary = {
    'model': CFG['model_id'],
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'probe': {
        'n_questions': CFG['n_questions'],
        'difficulty': CFG['difficulty_filter'],
        'baseline_accuracy': float(outcomes.mean()),
        'n_correct': int(outcomes.sum()),
        'n_wrong': int((1 - outcomes).sum()),
        'avg_tokens_generated': float(np.mean(n_tok_gen)),
    },
    'layer_analysis': {
        'top10_d_per_layer': layer_top10_d.tolist(),
        'mean_d_top10_per_layer': layer_top10_mean_d.tolist(),
        'peak_layer': peak_layer,
        'peak_is_gated_attn': peak_layer in CFG['gated_attn_layers'],
        'peak_depth_pct': peak_layer / CFG['n_layers'] * 100,
        'peak_top10_d_sum': float(layer_top10_d[peak_layer]),
    },
    'logit_lens': {
        'reasoning_tokens': list(tid_flat.keys()),
        'prob_per_layer': layer_reasoning_prob.tolist(),
        'commit_layer': commit_layer,
    },
    'decision': {
        'sae_target_layer': sae_target,
        'reason': (
            'peak is Gated Attn, train SAE there' if peak_layer in CFG['gated_attn_layers']
            else f'peak L{peak_layer} is GDN, SAE at nearest downstream Gated Attn L{sae_target}'
            if sae_target > peak_layer and sae_target in CFG['gated_attn_layers']
            else f'peak L{peak_layer} is GDN, SAE at L{sae_target} (peak+2 pattern)'
        ),
    },
}

with open(DRIVE_ROOT / 's4_l0_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
print(f'\n✅ Saved: {DRIVE_ROOT}/s4_l0_summary.json')